# 발표 캡처용 실행 노트북

실제 프로젝트 함수에 **시연용 가상 데이터**를 넣어 동작을 보여줍니다. 실제 회사 문서, 실서비스 검색 결과, 성능 측정 데이터가 아닙니다.

- 본문용: **CAP-01 표 파싱 / CAP-02 청킹 / CAP-03 문맥 재조립**
- 보조용: **CAP-04 RRF / CAP-05 멀티턴 검색어**
- 외부 API·Elasticsearch·LLM을 호출하지 않으며, 색인·DB를 변경하지 않습니다.
- 저장된 출력만 열어 캡처할 수 있습니다. 재실행은 저장소 루트 또는 notebooks 디렉터리에서 이 저장소의 Python 환경으로 실행하세요.
- 필요 패키지: `pip install -r notebooks/requirements-presentation.txt`
- 셀을 위에서 아래로 실행하고, CAP 셀의 **출력 영역**만 캡처하세요. 제목과 '시연용' 표기를 포함합니다.
- 설명: [캡처 안내](../docs/presentation/CAPTURE_GUIDE.md) / [슬라이드 원고](../docs/presentation/SLIDE_CONTENT.md)


In [1]:
from pathlib import Path
import sys
from html import escape
from importlib.metadata import version
from IPython.display import HTML, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "ai-server/app/parser/confluence_parser.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("저장소 루트 또는 notebooks 디렉터리에서 실행하세요.")
if str(ROOT / "ai-server") not in sys.path:
    sys.path.insert(0, str(ROOT / "ai-server"))

from app.parser.confluence_parser import parse_confluence_html, split_text_into_chunks
from app.retrieval.es_client import _group_chunk_texts, _rrf_scores
from app.retrieval.query_builder import build_search_query

# config 모듈이 .env를 읽을 수 있지만, 설정값/인증정보는 출력하지 않습니다.
# 아래 시연은 순수 계산 helper만 호출합니다. ES client 생성이나 API 호출은 없습니다.
print("실제 프로젝트 함수 로드 완료 · 외부 서비스 호출 없음")
print("실행 패키지:", {p: version(p) for p in [
    "beautifulsoup4", "langchain-text-splitters", "elasticsearch", "pydantic-settings"]})


실제 프로젝트 함수 로드 완료 · 외부 서비스 호출 없음
실행 패키지: {'beautifulsoup4': '4.15.0', 'langchain-text-splitters': '0.3.11', 'elasticsearch': '8.19.3', 'pydantic-settings': '2.11.0'}


In [2]:
# 출력 전용 HTML. 입력은 가상 데이터이며 일반 텍스트는 반드시 escape합니다.
STYLE = """
<style>
.capture {font-family:Arial,'Noto Sans CJK KR','Malgun Gothic',sans-serif;background:#fff;
 color:#16212d;padding:24px;max-width:1100px;border:1px solid #dbe2e8;line-height:1.65}
.capture h2 {font-size:23px;color:#16212d;margin:0 0 6px}
.capture h3 {font-size:17px;color:#16212d;margin:8px 0}
.capture .label {font-size:12px;color:#526172;margin-bottom:18px}
.capture .cols {display:flex;gap:20px;align-items:flex-start}
.capture .panel {flex:1;min-width:0;background:#f6f8fa;padding:14px}
.capture table {border-collapse:collapse;width:100%;font-size:14px;color:#16212d;margin:8px 0}
.capture th,.capture td {border:1px solid #cbd5df;padding:8px;text-align:left;background:#fff}
.capture th {background:#e9eff5;font-weight:bold}
.capture pre {font-size:13px;background:#fff;color:#16212d;padding:12px;white-space:pre-wrap;
 overflow-wrap:anywhere;border:1px solid #dbe2e8;line-height:1.65}
.capture mark {background:#fff0a6;color:#16212d;padding:1px}
.capture .note {font-size:13px;color:#34465a;margin-top:16px}
</style>
"""

def table(headers, rows):
    head = "<tr>" + "".join(f"<th>{escape(str(h))}</th>" for h in headers) + "</tr>"
    body = "".join("<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in row) + "</tr>" for row in rows)
    return f"<table>{head}{body}</table>"

def panel(title, body):
    return f'<div class="panel"><h3>{escape(title)}</h3>{body}</div>'

def pre(text):
    return f"<pre>{escape(text)}</pre>"

def capture(asset_id, title, body, note):
    display(HTML(STYLE + f'<section class="capture"><h2>{asset_id} · {escape(title)}</h2>'
                 '<div class="label">시연용 가상 데이터 · 실제 프로젝트 함수 실행 · 실서비스 성능 측정 아님</div>'
                 + body + f'<div class="note">{escape(note)}</div></section>'))


## CAP-01 · 병합 표 파싱 — 본문 3장

원본 표에는 rowspan과 colspan이 있습니다. 오른쪽은 `parse_confluence_html()`의 실제 반환 텍스트입니다. Markdown을 다시 표로 렌더링하지 않아, 검색에 쓰일 텍스트 표현 자체를 볼 수 있습니다.

코드: `ai-server/app/parser/confluence_parser.py` → `_table_to_records` → `_table_records_to_markdown`.


In [3]:
sample_html = """
<table>
<tr><th>팀</th><th>업무</th><th>담당</th><th>안내</th></tr>
<tr><td rowspan="2">운영팀</td><td>계정 발급</td><td>김담당</td><td>신청서 제출</td></tr>
<tr><td>권한 변경</td><td>이담당</td><td>팀장 승인</td></tr>
<tr><td>공통</td><td colspan="2">월간 점검 안내</td><td>공지 확인</td></tr>
</table>
"""
parsed = parse_confluence_html(sample_html, metadata={"source": "synthetic-demo"})
parsed_text = parsed["cleaned_text"]
assert "| 운영팀 | 권한 변경 | 이담당 | 팀장 승인 |" in parsed_text
assert "| 공통 | 월간 점검 안내 | 월간 점검 안내 | 공지 확인 |" in parsed_text
capture("CAP-01", "병합 셀의 행·열 관계를 텍스트로",
        '<div class="cols">' + panel("입력 · 병합된 HTML 표", sample_html)
        + panel("출력 · 파싱된 Markdown 텍스트", pre(parsed_text)) + "</div>",
        "운영팀은 다음 행에도 채워지고, 두 열에 걸친 공통 안내는 각 열에 반복됩니다. 파싱 효과의 정량 평가가 아니라 변환 동작 시연입니다.")


## CAP-02 · 청크 경계와 overlap — 본문 3장 보조

프로젝트 기본값인 800자·overlap 150자를 그대로 사용합니다. 읽기 쉬운 가상 업무 안내를 충분히 길게 만들고, 인접 청크의 실제 접미사·접두사 공통 문자열을 강조합니다. 설정 150자와 실제 겹치는 글자 수는 다를 수 있습니다.


In [4]:
sections = [
    "신규 계정은 사용 목적과 담당 팀을 확인한 뒤 신청서를 작성하고 승인 결과를 안내받습니다.",
    "권한 변경은 필요한 메뉴와 업무 범위를 적고 팀장 승인을 받은 뒤 담당자에게 전달합니다.",
    "문서 검색은 구체적인 업무명으로 시작하고 답변에 첨부된 원문 링크에서 세부 조건을 확인합니다.",
    "월간 점검 일정은 팀 공지에서 확인하며 변경된 안내는 업무를 시작하기 전에 다시 읽습니다.",
]
long_text = "\n".join(f"안내 {i:02d}. {sections[(i-1) % len(sections)]}" for i in range(1, 41))
chunks = split_text_into_chunks("demo-guide", "가상 업무 안내", long_text,
                               metadata={"source": "synthetic-demo"},
                               chunk_size=800, chunk_overlap=150)
assert len(chunks) >= 3
assert all(c["chunk_index"] == i and c["total_chunks"] == len(chunks)
           and len(c["text"]) <= 800 for i, c in enumerate(chunks))

def actual_overlap(left, right):
    return next((left[-n:] for n in range(min(len(left), len(right)), 0, -1)
                 if left[-n:] == right[:n]), "")

left, right = chunks[0]["text"], chunks[1]["text"]
overlap = actual_overlap(left, right)
assert 0 < len(overlap) <= 150
# 출력만 발췌합니다. 청킹 입력이나 실제 함수 결과를 자르거나 수정하지 않습니다.
left_view = escape(left[-240:-len(overlap)]) + "<mark>" + escape(overlap) + "</mark>"
right_view = "<mark>" + escape(overlap) + "</mark>" + escape(right[len(overlap):240])
summary = table(["chunk_id", "doc_id", "순서", "전체 청크 수", "글자 수"],
                [[c["chunk_id"], c["doc_id"], c["chunk_index"], c["total_chunks"], len(c["text"])]
                 for c in chunks[:3]])
capture("CAP-02", "청크를 나누되, 인접 문맥을 일부 겹치기", summary + '<div class="cols">'
        + panel("청크 0 · 끝부분 발췌", "<pre>…\n" + left_view + "</pre>")
        + panel("청크 1 · 시작부분 발췌", "<pre>" + right_view + "\n…</pre>") + "</div>",
        f"설정: 800자 / overlap 150자 · 이번 경계의 실제 공통 부분: {len(overlap)}자(노란색). "
        f"총 {len(chunks)}개 중 처음 3개 메타데이터와 두 청크의 경계만 표시했습니다.")


chunk_id,doc_id,순서,전체 청크 수,글자 수
demo-guide_chunk_0,demo-guide,0,4,756
demo-guide_chunk_1,demo-guide,1,4,756
demo-guide_chunk_2,demo-guide,2,4,758


## CAP-03 · 검색된 청크에서 문서 문맥으로 — 본문 4장

검색에 걸렸다고 **가정한** 한 청크와, 같은 문서의 정렬된 청크를 `_group_chunk_texts()`에 넣은 결과를 비교합니다. ES 후보 검색·최종 문서 선택은 실행하지 않습니다.

코드: `ai-server/app/retrieval/es_client.py`. 실제 ES 조회는 doc_id, chunk_index 순으로 정렬하며 helper 자체는 정렬하지 않습니다. overlap 중복 제거도 하지 않습니다.


In [5]:
demo_parts = [
    "기능 안내: 가상 업무 도우미는 사내 문서 검색과 원문 링크 안내를 지원합니다. 공통 안내는 원문에서 확인합니다.",
    "공통 안내는 원문에서 확인합니다. 설치 안내: 포털에서 계정을 신청한 뒤 담당 팀의 접근 권한을 받습니다.",
    "이용 안내: 질문에 업무명을 포함하고, 생성 답변의 세부 조건은 연결된 문서에서 확인합니다.",
]
sorted_hits = [{"_source": {"doc_id": "demo-helper", "chunk_index": i, "text": text}}
               for i, text in enumerate(demo_parts)]
matched_chunk = demo_parts[1]
assembled = _group_chunk_texts(sorted_hits, max_chars=3000)["demo-helper"]
assert assembled == "\n".join(demo_parts)
assert assembled.count("공통 안내는 원문에서 확인합니다.") == 2
capture("CAP-03", "검색된 한 조각에서 주변 문맥까지", '<div class="cols">'
        + panel("검색에 걸렸다고 가정한 청크 1", pre(matched_chunk))
        + panel("정렬된 청크 0 → 1 → 2 재조립", pre(assembled)) + "</div>",
        "가상 청크 3개로 재조립 helper만 실행했습니다. 기능 설명까지 포함되지만 겹치는 문장은 그대로 남습니다. 원본 전체 복원은 아닙니다.")

# 화면에 크게 넣지 않는 동작 검증: 3,000자는 청크 추가 중단 기준입니다.
limit_hits = [{"_source": {"doc_id": "limit-demo", "text": "가" * 800}} for _ in range(5)]
limit_text = _group_chunk_texts(limit_hits, max_chars=3000)["limit-demo"]
assert len(limit_text) == 3203  # 800자 청크 4개 + 청크 사이 개행 3개


## CAP-04 · RRF 순위 결합 — 보조 자료

후보 순위는 설명용으로 만든 값입니다. 실제 `_rrf_scores()`를 두 목록에 각각 실행하고 합칩니다. `k=60`을 명시해 환경변수와 무관하게 예시를 재현합니다. 최신성 가산·ES 검색·문서 중복 제거는 이 시연 범위 밖입니다.


In [6]:
def mock_hits(ids):
    return [{"_source": {"chunk_id": chunk_id}} for chunk_id in ids]

bm25_order = ["A", "B", "C"]
knn_order = ["C", "D", "B"]
bm25_scores = _rrf_scores(mock_hits(bm25_order), k=60)
knn_scores = _rrf_scores(mock_hits(knn_order), k=60)
combined = {key: bm25_scores.get(key, 0) + knn_scores.get(key, 0)
            for key in sorted(set(bm25_scores) | set(knn_scores))}
ranked = sorted(combined, key=lambda key: combined[key], reverse=True)
assert ranked == ["C", "B", "A", "D"]
assert abs(combined["C"] - (1/63 + 1/61)) < 1e-12

def position(order, key):
    return order.index(key) + 1 if key in order else "후보 밖"

rows = [[i, key, position(bm25_order, key), position(knn_order, key),
         f"{bm25_scores.get(key, 0):.6f}", f"{knn_scores.get(key, 0):.6f}", f"{combined[key]:.6f}"]
        for i, key in enumerate(ranked, 1)]
capture("CAP-04", "서로 다른 검색을 순위로 결합",
        table(["결합 순위", "청크", "BM25 순위", "kNN 순위", "BM25 기여", "kNN 기여", "RRF 합"], rows),
        "각 기여 = 1 / (60 + 순위), 후보 밖이면 0. C는 두 목록의 지지를 받아 A보다 앞섭니다. "
        "가상 후보 예시이며 이 점수는 정답 확률이 아닙니다. 최신성 가산 전입니다.")


결합 순위,청크,BM25 순위,kNN 순위,BM25 기여,kNN 기여,RRF 합
1,C,3,1,0.015873,0.016393,0.032266
2,B,2,3,0.016129,0.015873,0.032002
3,A,1,후보 밖,0.016393,0.000000,0.016393
4,D,후보 밖,2,0.000000,0.016129,0.016129


## CAP-05 · 멀티턴 검색어 보정 — 보조 자료

`build_search_query()`의 현재 동작은 직전 사용자 발화를 문자열로 붙이는 것입니다. LLM으로 문장을 재작성하거나 답변 품질을 평가하지 않습니다. 답변 생성에는 원래 질문과 대화 이력이 별도로 사용됩니다.


In [7]:
history = [
    {"role": "user", "content": "교육비 지원 신청 절차 알려줘"},
    {"role": "assistant", "content": "시연용 이전 답변입니다."},
]
follow_up = "그건 언제까지 신청해야 해?"
search_query = build_search_query(follow_up, history, turns=1)
assert search_query == "교육비 지원 신청 절차 알려줘 그건 언제까지 신청해야 해?"
assert build_search_query(follow_up, history, turns=0) == follow_up
assert "시연용 이전 답변" not in search_query
capture("CAP-05", "후속 질문에 이전 주제어를 보충",
        table(["구분", "문자열"], [
            ["직전 사용자 발화", history[0]["content"]],
            ["현재 질문", follow_up],
            ["검색에 넣는 문자열", search_query],
        ]),
        "실제 build_search_query(..., turns=1) 출력. LLM 재작성·실제 검색·답변 생성은 실행하지 않았습니다. "
        "주제가 바뀐 경우에도 앞 발화를 붙일 수 있다는 한계가 있습니다.")


구분,문자열
직전 사용자 발화,교육비 지원 신청 절차 알려줘
현재 질문,그건 언제까지 신청해야 해?
검색에 넣는 문자열,교육비 지원 신청 절차 알려줘 그건 언제까지 신청해야 해?


## 실행 확인

아래 셀까지 실행되면 앞선 예시들의 assert가 통과한 것입니다. 이는 시연 입력에 대한 함수 동작 확인이며, 전체 서비스 테스트나 성능 평가를 대신하지 않습니다. 평가 캡처는 별도로 Langfuse에서 준비합니다.


In [8]:
print("CAP-01~05 실행 및 예시 검증 완료")
print("외부 API 호출 없음 · 실제 회사 데이터 없음 · 색인/DB 변경 없음")


CAP-01~05 실행 및 예시 검증 완료
외부 API 호출 없음 · 실제 회사 데이터 없음 · 색인/DB 변경 없음
